# 把前端项目发布到公网

## Next.js 项目为什么不能直接交给 Nginx

上一节构建后，.next/server/app/ 里虽然能看到 index.html 和 text-lab.html，但不能像 Vite 的 dist/ 那样直接交给 Nginx。

原因是 .next/ 不是一个可以直接发布的干净网站目录：

- 页面引用的 CSS、JavaScript 分散在 .next/static/ 等其他目录。
- 没有后缀的 /text-lab 应该对应哪个文件、404 如何处理、缓存头如何设置，这些工作需要 Next 服务运行时处理。
- 目录中还包含 RSC 数据、manifest 和服务端 JavaScript 等只供 Next 使用的文件。

.next/ 是给 start 这台 Node 服务使用的构建结果，不是专门给 Nginx 直接发送的静态网站目录。

## 第一条路：运行常驻的 Next 服务

在服务器上启动 Next 自带的 Node 服务：

```bash
npm run start
```

Next 服务会接收用户请求，并使用 .next/ 中的构建结果响应页面。它最大的能力是：对于动态页面，可以在服务器收到请求时根据最新数据生成页面。

这也是 Next.js 默认、并且在 Vercel 等平台上常见的部署方式。

### 优点

- 可以在服务器上按请求生成动态 HTML。
- 可以连接数据库、读取服务器文件和使用服务器端密钥。
- 可以在同一个 Next 项目中提供 API 路由。

### 代价

- 服务器必须有一个 Node 进程长期运行，通常需要进程管理器或服务管理工具保证它自动重启。
- 服务器需要承担 Node 服务的运行资源和安全维护。

## 第二条路：静态导出 out/

如果每一页在构建时就可以确定，可以告诉 Next.js 不需要根据请求动态生成 HTML ，直接把页面导出成一个干净的静态网站目录。

在项目里加一行：`output: "export"`

执行 npm run build 后会生成 out/：

```text
out/
├── index.html            /
├── text-lab.html         /text-lab
├── 404.html
└── _next/
    └── static/           打包后的 CSS、JavaScript 等资源
```

out/ 中的页面已经预渲染成 HTML。需要交互的客户端组件还会带上浏览器端 JavaScript，因此输入、导航高亮和动画仍然可以运行。

静态导出的限制是：HTML 在构建时就确定了，不能依靠 Next 服务在每次请求到来时重新计算页面。
需要动态内容时，可以让浏览器加载静态页面后，再通过后端 API 获取数据。

## 怎么选：

### 适合静态导出的情况

- 纯展示站、博客、文档和作品集。
- 页面结构在构建时就能确定。
- 使用“静态前端 + 后端 API”，由浏览器在页面加载后请求动态数据。
- 希望服务器只提供静态文件，不长期运行 Node 服务。

### 适合常驻服务的情况

- 页面 HTML 必须在服务器收到请求时根据最新数据生成。
- 动态内容本身还需要被搜索引擎直接收录。
- 需要在 Next 服务端使用数据库、文件或服务器密钥。

登录、注册、实时数据和数据库读写，很多时候都可以由静态前端加载后调用后端 API 完成。后续也采用“静态前端 + 独立后端 API”的分工的方式

## 静态导出发布：增加配置

打开项目中的 next.config.mjs。原来可能是：

```javascript
const nextConfig = {};

export default nextConfig;
```

改成：

```javascript
const nextConfig = {
  output: "export",
};

export default nextConfig;
```

output: "export" 告诉 Next.js：构建完成后，把每一页预渲染成静态 HTML，并把资源一起放入 out/。

out/ 是构建产物，不应该提交到 Git。项目的 .gitignore 至少应包含：

```gitignore
.next/
out/
node_modules/
```

配置修改后，本地先构建一次：

```bash
npm run build
```

确认 out/index.html 和 out/text-lab.html 都已经生成。

## 静态导出发布：Nginx 指向 out/

服务器端流程和 4.4 基本一致

pull - install - build -修改默认路径

但 Nginx 的根目录从 dist/ 改成 out/，并增加一条 try_files 规则。

进人服务器后编辑配置：

```bash
sudo vim /etc/nginx/sites-enabled/default
```

示例配置：

```nginx
server {
    listen 80 default_server;
    server_name _;

    root /home/ubuntu/zero-to-tech/out;
    index index.html;

    location / {
        # /text-lab 没有 .html 后缀，自动尝试 text-lab.html
        try_files $uri $uri.html $uri/ =404;
    }
}
```

try_files 会依次尝试：

1. 直接查找 URL 对应的文件或目录。
2. 找不到时给 URL 加 .html 后缀再查找。
3. 仍然找不到就返回 404。

因此访问 /text-lab 时，Nginx 会找到 out/text-lab.html。

然后：

```bash
sudo nginx -t
sudo systemctl reload nginx
```

## 我们没走的那条路：A

如果服务器上运行常驻的 Next 服务，服务端组件可以：

- 直接连接数据库。
- 读取服务器上的文件。
- 使用只有服务器知道的密钥。
- 在 app/api/... 中提供后端接口。
- 根据每个请求生成动态页面。

这样前端和后端可以放在同一个 Next 项目中，这就是常说的“Next 全栈”。

本课程暂时采用：

```text
静态前端：Next.js 导出 out/，由 Nginx 提供
独立后端：使用 Python 单独运行服务，提供情感分数和拼音等 API
两者之间：通过 HTTP API 通信
```